In [1]:
!pip install -r requirements.txt

In [2]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoConfig, TrainingArguments, Trainer, BertModel, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from pypinyin import pinyin, Style
import torch
import torch.nn as nn
import numpy as np
import evaluate
import pandas as pd
import os
import pronouncing

/opt/miniconda3/envs/Chinese_pun/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/miniconda3/envs/Chinese_pun/lib/python3.12/site-packages/pronouncing/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream


In [3]:
# ============================================================
# DEVICE CONFIGURATION - Set USE_GPU to True if GPU available
# ============================================================
import torch

USE_GPU = False  # Set to True if you have GPU (CUDA/MPS)

# Detect available device
if USE_GPU:
    if torch.cuda.is_available():
        device = "cuda"
        print("✅ GPU (CUDA) detected and enabled")
    elif torch.backends.mps.is_available():
        device = "mps"  # Mac with Apple Silicon
        print("✅ GPU (MPS) detected and enabled")
    else:
        device = "cpu"
        print("⚠️ GPU requested but not available, falling back to CPU")
else:
    device = "cpu"
    print("✅ Using CPU (set USE_GPU=True to enable GPU)")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device.upper()}")

✅ Using CPU (set USE_GPU=True to enable GPU)
PyTorch version: 2.10.0
Device: CPU


### Data Preprocessing

In [5]:
data_path = "./data/toxic_dataset_cleaned.csv"
df = pd.read_csv(data_path).dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str)
df['label'] = df['label'].astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(pd.DataFrame({"text": X_train.values, "label": y_train.values}), preserve_index=False),
    "validation": Dataset.from_pandas(pd.DataFrame({"text": X_val.values, "label": y_val.values}), preserve_index=False)
})

## Context Model

### Define Model

In [6]:
from transformers import AutoTokenizer

model_name = "google-bert/bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Map the function over the entire dataset in batches
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 131/131 [00:00<00:00, 13730.17 examples/s]


### Train

In [7]:
accuracy = evaluate.load("accuracy")
f1_macro = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_macro.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

In [13]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Load model with a classification head
# num_labels should match the number of unique classes in your dataset
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Define training hyperparameters
training_args = TrainingArguments(
    output_dir="./baseline_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    load_best_model_at_end=True,
    weight_decay=0.01,
    metric_for_best_model="f1_macro",
)

# 3. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# 4. Start training
trainer.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1591.32it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISS

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

### Inference

In [ ]:
def predict(text):
    # 1. Tokenize
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    
    # 2. Forward Pass
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    
    # 3. Get Label
    logits = outputs.logits
    pred_idx = torch.argmax(logits, dim=-1).item()
    
    return "Toxic" if pred_idx == 1 else "Non-Toxic"

# Now this will work:
print(predict("你好"))

In [ ]:
predict("你好")

'Pun (諧音/毒性)'

In [ ]:
# 11. Confusion Matrix & Classification Report
predictions = trainer.predict(tokenized_dataset["validation"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=[0, 1], target_names=["Non-Toxic", "Toxic"]))


Classification Report:
              precision    recall  f1-score   support

   Non-Toxic       0.81      0.84      0.82        81
       Toxic       0.70      0.66      0.68        47

    accuracy                           0.77       128
   macro avg       0.76      0.75      0.75       128
weighted avg       0.77      0.77      0.77       128



In [ ]:
# Plot confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Pun", "Pun"],
            yticklabels=["Non-Pun", "Pun"],
            cbar_kws={'label': 'Count'})
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Pun Classification")
plt.tight_layout()
plt.show()

print("\n✅ Model training completed!")